# 05 — Delivery Delay Analysis
**Random Forest + SMOTE + Threshold Tuning (v3)**

Fixed: class imbalance (93:7), added peak_season/weekend features. Late recall: 2% -> 76%.

## Setup

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sqlalchemy import create_engine, text
import sys, os, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, os.path.join(os.getcwd(), '..'))
from config import PG_URL
engine = create_engine(PG_URL)
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('Blues_d')
print("Setup complete")

import joblib
from config import MODELS_DIR
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import confusion_matrix, roc_curve, auc, classification_report, ConfusionMatrixDisplay
from imblearn.over_sampling import SMOTE


## 1. Delivery overview by state

In [ ]:

sql = '''SELECT c.state, COUNT(*) AS total_orders,
SUM(CASE WHEN f.delivery_delay_days>0 THEN 1 ELSE 0 END) AS late_orders,
ROUND(AVG(f.delivery_delay_days)::numeric,1) AS avg_delay_days,
ROUND(SUM(CASE WHEN f.delivery_delay_days>0 THEN 1.0 ELSE 0 END)/COUNT(*)*100,1) AS late_pct
FROM olist.fact_orders f JOIN olist.dim_customers c ON f.customer_id=c.customer_id
WHERE f.is_delivered=1 AND f.delivery_delay_days IS NOT NULL
GROUP BY c.state ORDER BY late_pct DESC LIMIT 15'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
fig,axes = plt.subplots(1,2,figsize=(14,5))
axes[0].barh(df['state'][::-1],df['late_pct'][::-1],color='steelblue')
axes[0].set_title('Late Delivery % by State'); axes[0].set_xlabel('Late %')
axes[1].barh(df['state'][::-1],df['avg_delay_days'][::-1],color='darkorange')
axes[1].set_title('Avg Delay Days by State'); axes[1].set_xlabel('Days')
plt.tight_layout(); plt.show()


## 2. Delay by month

In [ ]:

sql = '''SELECT order_month, ROUND(AVG(delivery_delay_days)::numeric,1) AS avg_delay,
ROUND(SUM(CASE WHEN delivery_delay_days>0 THEN 1.0 ELSE 0 END)/COUNT(*)*100,1) AS late_pct
FROM olist.fact_orders WHERE is_delivered=1 AND delivery_delay_days IS NOT NULL
GROUP BY order_month ORDER BY order_month'''
with engine.connect() as conn:
    df_m = pd.read_sql(text(sql),conn)
months = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
df_m['month_name'] = df_m['order_month'].apply(lambda x: months[x-1])
fig,axes = plt.subplots(1,2,figsize=(13,4))
axes[0].bar(df_m['month_name'],df_m['avg_delay'],color='steelblue')
axes[0].set_title('Avg Delay by Month'); axes[0].set_ylabel('Days')
axes[1].bar(df_m['month_name'],df_m['late_pct'],color='darkorange')
axes[1].set_title('Late % by Month'); axes[1].set_ylabel('%')
plt.tight_layout(); plt.show()
print("Nov-Dec (peak_season) tend to have higher delays due to holiday rush")


## 3. Model performance — before vs after fix

In [ ]:

model     = joblib.load(os.path.join(MODELS_DIR,'delivery_model.pkl'))
le_cust   = joblib.load(os.path.join(MODELS_DIR,'delivery_le_cust.pkl'))
le_seller = joblib.load(os.path.join(MODELS_DIR,'delivery_le_seller.pkl'))
sql = '''SELECT f.price,f.freight_value,f.order_month,f.order_quarter,f.order_year,
f.payment_installments,f.state AS customer_state,s.seller_state,
p.weight_g,p.length_cm,p.height_cm,p.width_cm,
CASE WHEN s.seller_state!=f.state THEN 1 ELSE 0 END AS cross_state,
ROUND((f.freight_value/NULLIF(f.price+f.freight_value,0))::numeric,4) AS freight_ratio,
ROUND((NULLIF(p.length_cm,0)*NULLIF(p.height_cm,0)*NULLIF(p.width_cm,0))::numeric,2) AS volume_cm3,
CASE WHEN f.order_month IN (11,12) THEN 1 ELSE 0 END AS peak_season,
CASE WHEN EXTRACT(DOW FROM f.order_date::date) IN (0,6) THEN 1 ELSE 0 END AS weekend_order,
CASE WHEN f.delivery_delay_days>0 THEN 1 ELSE 0 END AS is_late
FROM olist.fact_orders f JOIN olist.dim_products p ON f.product_id=p.product_id
JOIN olist.dim_sellers s ON f.seller_id=s.seller_id
WHERE f.is_delivered=1 AND f.delivery_delay_days IS NOT NULL'''
with engine.connect() as conn:
    df = pd.read_sql(text(sql),conn)
df['customer_state_enc'] = le_cust.transform(df['customer_state'].fillna('SP').apply(lambda x: x if x in le_cust.classes_ else le_cust.classes_[0]))
df['seller_state_enc']   = le_seller.transform(df['seller_state'].fillna('SP').apply(lambda x: x if x in le_seller.classes_ else le_seller.classes_[0]))
features = ['price','freight_value','freight_ratio','order_month','order_quarter','order_year',
            'payment_installments','customer_state_enc','seller_state_enc','cross_state',
            'weight_g','length_cm','height_cm','width_cm','volume_cm3','peak_season','weekend_order']
X = df[features].fillna(0); y = df['is_late']
_,X_test,_,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]
fig,axes = plt.subplots(1,2,figsize=(13,5))
ConfusionMatrixDisplay(confusion_matrix(y_test,y_pred),display_labels=['On-time','Late']).plot(ax=axes[0],cmap='Blues',colorbar=False)
axes[0].set_title('Confusion Matrix — Delivery v3')
fpr,tpr,_ = roc_curve(y_test,y_proba)
axes[1].plot(fpr,tpr,color='steelblue',lw=2,label=f'AUC={auc(fpr,tpr):.3f}')
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('ROC Curve — Delivery v3'); axes[1].legend()
plt.tight_layout(); plt.show()
print(classification_report(y_test,y_pred,target_names=['On-time','Late']))


## 4. Feature importance

In [ ]:

importance = pd.DataFrame({'feature':features,'importance':model.feature_importances_}).sort_values('importance',ascending=True)
plt.figure(figsize=(10,7))
plt.barh(importance['feature'],importance['importance'],color='steelblue')
plt.title('Feature Importance — Delivery Delay Model v3')
plt.tight_layout(); plt.show()
